# Train Confidant's writing model on your writing

This teaches the writing model (Mistral Nemo 12B Base, the same one Confidant already uses) to write like you. It runs on Google's free graphics card and takes about an hour.

**Before you start:** in the menu, choose **Runtime → Change runtime type → T4 GPU → Save**.

Then run each step below **in order**: click the ▶ button at its top left and wait for it to finish (a green tick) before the next one. Keep this tab open the whole time; if Colab asks whether you're still there, click yes.

Nothing here is shared: your writing goes only into this temporary Colab machine, which is wiped when you close it.

## Step 1: Get the tools ready (about 3 minutes)

In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), "No graphics card. Choose Runtime → Change runtime type → T4 GPU, then run this step again."
print("Graphics card:", torch.cuda.get_device_name(0))
!pip install -q unsloth python-docx pypdf
print("Ready.")

## Step 2: Upload your writing

Click **Choose Files** and pick `my-writing.txt` (from Confidant: **Book → Samples → Download my writing**). You can also add `.txt`, `.docx` or `.pdf` files of your own writing. Only add writing you wrote yourself, nothing an AI wrote.

In [ ]:
from google.colab import files
import io, re

uploaded = files.upload()

def read(name, data):
    low = name.lower()
    if low.endswith(".docx"):
        import docx
        return "\n\n".join(p.text for p in docx.Document(io.BytesIO(data)).paragraphs)
    if low.endswith(".pdf"):
        from pypdf import PdfReader
        return "\n\n".join(page.extract_text() or "" for page in PdfReader(io.BytesIO(data)).pages)
    return data.decode("utf-8", errors="ignore")

texts = []
for name, data in uploaded.items():
    t = read(name, data)
    t = re.sub(r"\r\n?", "\n", t)
    t = re.sub(r"[ \t]+\n", "\n", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    t = re.sub(r"\s*(—|–)\s*", ", ", t)  # no dashes, as in Confidant
    texts.append(t.strip())
    print(f"{name}: {len(t.split()):,} words")

writing = "\n\n* * *\n\n".join(texts)
total = len(writing.split())
print(f"\nTotal: {total:,} words.")
if total < 20000:
    print("That works, but 20,000+ words teaches your voice much better. Intimate or action scenes you've written help it with those scenes too.")

## Step 3: Load the writing model (about 5 minutes)

In [ ]:
from unsloth import FastLanguageModel

SEQ = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Mistral-Nemo-Base-2407-bnb-4bit",
    max_seq_length=SEQ,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=7,
)
print("Model loaded.")

## Step 4: Train on your writing (30 to 60 minutes)

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

# Cut the writing into overlapping pieces of about 1,000 tokens.
ids = tokenizer(writing, add_special_tokens=False)["input_ids"]
step = SEQ - 128
pieces = [tokenizer.decode(ids[i:i + SEQ]) for i in range(0, max(1, len(ids) - 128), step)]
data = Dataset.from_dict({"text": pieces})

# Small collections need more passes; large ones fewer.
epochs = min(6, max(2, round(240 / max(1, len(pieces)))))
print(f"{len(pieces)} pieces, {epochs} passes.")

config = SFTConfig(
        dataset_text_field="text",
        packing=False,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=epochs,
        learning_rate=1e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        optim="adamw_8bit",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        output_dir="training",
        save_strategy="no",
        report_to="none",
        seed=7,
)
# Newer versions of the training library renamed one setting; handle both.
try:
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=data, args=config)
except TypeError:
    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=data, args=config)
trainer.train()
print("Training finished.")

## Step 5: Try it

This continues the last part of your writing. Read it: it should sound like you. Run it again for a different try.

In [ ]:
FastLanguageModel.for_inference(model)
start = writing[-1500:]
start = start[start.find(" ") + 1:]
inputs = tokenizer(start, return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=300, temperature=0.8, top_p=0.9, do_sample=True, repetition_penalty=1.05)
print("...", start[-300:])
print("\n----- written by your model -----\n")
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## Step 6: Download your voice file (about 5 minutes)

In [ ]:
import os, shutil, zipfile
model.save_pretrained("my-voice-lora")

# Turn it into the format Ollama uses. If this fails, the zip still holds the
# original files, which Ollama can usually use too.
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q ./llama.cpp/gguf-py mistral-common sentencepiece
r = subprocess.run(["python", "llama.cpp/convert_lora_to_gguf.py", "my-voice-lora",
                    "--base-model-id", "unsloth/Mistral-Nemo-Base-2407",
                    "--outfile", "my-voice.gguf", "--outtype", "f16"], capture_output=True, text=True)
gguf = os.path.exists("my-voice.gguf")
print("Converted for Ollama." if gguf else "Conversion didn't work, using the original files:\n" + r.stderr[-1500:])

os.makedirs("my-voice", exist_ok=True)
if gguf:
    shutil.copy("my-voice.gguf", "my-voice/my-voice.gguf")
    adapter = "./my-voice.gguf"
else:
    shutil.copytree("my-voice-lora", "my-voice/adapter", dirs_exist_ok=True)
    adapter = "./adapter"
with open("my-voice/Modelfile", "w") as f:
    f.write("FROM hf.co/mradermacher/Mistral-Nemo-Base-2407-GGUF:Q4_K_M\n")
    f.write(f"ADAPTER {adapter}\n")
    f.write("TEMPLATE {{ .Prompt }}\n")
    f.write("PARAMETER num_ctx 8192\n")
shutil.make_archive("my-voice", "zip", ".", "my-voice")
print(f"my-voice.zip: {os.path.getsize('my-voice.zip') / 1e6:.0f} MB")
files.download("my-voice.zip")

## Step 7: Add it to Ollama on your laptop

1. In your Downloads folder, right-click **my-voice.zip → Extract All → Extract**.
2. Open **Command Prompt** and run these two lines:

```
cd %USERPROFILE%\Downloads\my-voice\my-voice
ollama create my-voice -f Modelfile
```

3. In Confidant, open **Settings**, set **Writing model in Ollama** to `my-voice`, and tap **Test connection**. It should say the writing model is ready.

That's it: "Write…" requests, pasted passages, the Book tab and "Rewrite in my voice" all use your trained model from now on. You can close this Colab tab.